# Gemini Vision API - Structured Output

Este notebook demonstra como usar a Google Gemini API para análise de imagens com **Output Estruturado** (JSON), garantindo que a resposta siga um formato específico.

### Pré-requisitos
1. Ter o `uv` instalado.
2. Ter uma API Key do Google AI Studio.
3. Adicionar sua chave no arquivo `.env`.

In [ ]:
import os
import json
from pathlib import Path
import google.generativeai as genai
from dotenv import load_dotenv
from PIL import Image
from IPython.display import display
import typing_extensions as typing

# Carrega as variáveis de ambiente
load_dotenv()

# Configura a API Key
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key or api_key == "your_api_key_here":
    print("❌ ERRO: GOOGLE_API_KEY não encontrada no arquivo .env!")
else:
    genai.configure(api_key=api_key)
    print("✅ Gemini SDK configurado.")

## 1. Definir o Esquema da Resposta (Structured Output)
Usaremos uma classe Python para definir exatamente como queremos receber os dados.

In [ ]:
class ImageAnalysis(typing.TypedDict):
    categoria: str # Gato, Cachorro, Pássaro ou Outro
    objetos_detectados: list[str]
    confianca: float # De 0 a 1
    descricao_curta: str

## 2. Processar Imagem com Output Estruturado

In [ ]:
img_path = Path("imagens")
images = list(img_path.glob("*"))

index = 0 # Altere para testar outras imagens

if index < len(images):
    selected_image = images[index]
    print(f"Analisando: {selected_image.name}")
    
    img = Image.open(selected_image)
    display(img.reduce(4))

    # Configura o modelo com o esquema de resposta
    model = genai.GenerativeModel('gemini-2.5-flash')

    prompt = "Analise esta imagem e forneça as informações estruturadas. A categoria deve ser obrigatoriamente uma destas: Gato, Cachorro, Pássaro ou Outro."

    try:
        # O segredo está no generation_config com response_mime_type e response_schema
        response = model.generate_content(
            [prompt, img],
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json",
                response_schema=ImageAnalysis
            )
        )
        
        # Converte a string JSON de resposta em um dicionário Python
        res_data = json.loads(response.text)
        
        print("\n🚀 Resultado Estruturado (JSON):")
        print(json.dumps(res_data, indent=4, ensure_ascii=False))
        
        print(f"\n📍 Categoria Detectada: {res_data['categoria']}")
        print(f"📍 Descrição: {res_data['descricao_curta']}")
        
    except Exception as e:
        print(f"❌ Erro: {e}")
else:
    print("❌ Imagem não encontrada.")